ChatGroq API

In [1]:
from langchain_groq.chat_models import ChatGroq
import os

# Never hardcode API keys in code.
Groq_Token = os.getenv("CHAT_GROQ_API_KEY")

if not Groq_Token:
    raise ValueError("Set CHAT_GROQ_API_KEY in your environment before running this notebook.")

In [2]:
# Code block to check for available models using the Groq API (from https://console.groq.com/docs/models)
import requests

api_key = Groq_Token
url = "https://api.groq.com/openai/v1/models"

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json",
}

response = requests.get(url, headers=headers)

print(response.json())

{'object': 'list', 'data': [{'id': 'groq/compound', 'object': 'model', 'created': 1756949530, 'owned_by': 'Groq', 'active': True, 'context_window': 131072, 'public_apps': None, 'max_completion_tokens': 8192}, {'id': 'groq/compound-mini', 'object': 'model', 'created': 1756949707, 'owned_by': 'Groq', 'active': True, 'context_window': 131072, 'public_apps': None, 'max_completion_tokens': 8192}, {'id': 'llama-3.1-8b-instant', 'object': 'model', 'created': 1693721698, 'owned_by': 'Meta', 'active': True, 'context_window': 131072, 'public_apps': None, 'max_completion_tokens': 131072}, {'id': 'openai/gpt-oss-safeguard-20b', 'object': 'model', 'created': 1761708789, 'owned_by': 'OpenAI', 'active': True, 'context_window': 131072, 'public_apps': None, 'max_completion_tokens': 65536}, {'id': 'canopylabs/orpheus-arabic-saudi', 'object': 'model', 'created': 1765926439, 'owned_by': 'Canopy Labs', 'active': True, 'context_window': 4000, 'public_apps': None, 'max_completion_tokens': 50000}, {'id': 'met

In [3]:
# Groq LLM model initialization
model_name = "llama-3.1-8b-instant"
llm = ChatGroq(model=model_name, api_key=Groq_Token, temperature=0)

# Statement
sentence = "The product quality is amazing but the delivery was delayed. However I am happy with the customer service."

### Zero Shot

We ask the model to perform a task without providing any examples. The model relies entirely on its pre-trained knowledge and the clarity of the instruction.
- Clear instructions improve output quality
- Works well for simple tasks, may fail on complex tasks

In [4]:
# Zero-shot prompt
query = f"""
You are a sentiment analysis model.
Your task is to analyze the sentiment expressed in the given text and classify it as "positive", "negative", or "neutral".
Provide the sentiment label and a brief reason.

Sentence: {sentence}
"""

answer = llm.invoke(query)
print(answer.content)

Sentiment Label: Neutral

Reason: The sentence contains both positive and negative sentiments. The phrase "amazing" expresses a strong positive sentiment towards the product quality. However, the phrase "delivery was delayed" expresses a negative sentiment. The phrase "I am happy with the customer service" also expresses a positive sentiment. Since the sentence contains both positive and negative sentiments, the overall sentiment is classified as neutral.


### Few Shot

We provide a few examples to guide the model. This helps the model understand the expected pattern and improves accuracy.
- Demonstrates input-output format
- Reduces ambiguity

In [5]:
# Few-shot prompt
query = f"""
You are a sentiment analysis model.
Your task is to analyze the sentiment expressed in the given text and classify it as "positive", "negative", or "neutral".
Provide the sentiment label and a brief reason.

Here are a few examples:
1. Sentence: "The customer service was excellent, and I received my order quickly."
Sentiment: Positive

2. Sentence: "The food was bland and the service was slow."
Sentiment: Negative

3. Sentence: "The product is okay, but it's not worth the price."
Sentiment: Neutral

Sentence: {sentence}
"""

answer = llm.invoke(query)
print(answer.content)

Sentiment: Positive

Reason: The text mentions two negative aspects ("the delivery was delayed"), but they are outweighed by two positive aspects ("the product quality is amazing" and "I am happy with the customer service"). The overall tone is positive due to the emphasis on the product quality and the customer service.


### Role-based prompting

We assign a specific persona or role to the model. This influences tone, style, and sometimes reasoning approach.
- "You are X..." defines behavior
- Useful for creative, domain-specific, or stylistic tasks

In [6]:
model_name = "llama-3.3-70b-versatile"
llm = ChatGroq(model=model_name, api_key=Groq_Token, temperature=0)

In [7]:
# Without persona
query = "Write me a short poem."
answer = llm.invoke(query)
print(answer.content)

Here's a short poem:

The sun sets slow and paints the sky,
A fiery hue that makes me sigh.
The stars come out and twinkle bright,
A night of rest, a peaceful sight.

The world is calm, the darkness deep,
A time for dreams, a time to sleep.
The moon's soft glow, a gentle beam,
Guides me through, a peaceful dream.


In [8]:
# With persona
query = "You are Shakespeare, an English writer. Write me a short poem."
answer = llm.invoke(query)
print(answer.content)

Fairest maiden, with thy beauty's might,
Thou dost enthrall my heart, and senses bright.
As sunshine doth illume the morning dew,
Thy lovely face, my soul, doth forever renew.

Thy tresses, golden threads, that gently play,
Do dance upon thy shoulders, in a lovely sway.
Thy eyes, like sapphires, shining bright and blue,
Do sparkle with a fire, that my heart doth pursue.

Oh, fairest one, thou art a wondrous sight,
A treasure to behold, a pure delight.
In thy sweet presence, I am lost, yet found,
A captive to thy charm, forever bound.


### System Prompting

Instead of writing everything in one string, we separate instructions using system and user messages.
- System message defines behavior
- User message defines the task
- More structured and reliable

In [9]:
from langchain_core.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage(content="You are a helpful assistant that explains concepts in simple terms."),
    HumanMessage(content="Explain what a black hole is in 2 lines."),
]

response = llm.invoke(messages)
print(response.content)

A black hole is a region in space where gravity is so strong that nothing, including light, can escape once it gets too close. It's formed when a massive star collapses in on itself, creating an incredibly dense point with an intense gravitational pull.


### Structured Reasoning

Structured reasoning asks the model to think step by step before answering. In practice, ask it to reason internally and return a concise final answer plus a brief rationale (instead of full chain-of-thought).
- Encourages logical breakdown
- Improves accuracy on multi-step problems

In [10]:
query = """
Solve the problem.
Think step-by-step internally, then provide:
Final Answer:
Brief Rationale (1 sentence):

Problem: If a train travels 60 km in 1 hour, how far will it travel in 5 hours?
"""

response = llm.invoke(query)
print(response.content)

Final Answer: 300 km
Brief Rationale: To find the distance traveled in 5 hours, we multiply the speed of the train (60 km/h) by the time (5 hours), resulting in a total distance of 300 km.


### Output Format Control

In [11]:
# JSON
query = """
Extract information from the text and return ONLY valid JSON.

Text: "John bought a laptop for $1200 on 5th May 2026."

Format:
{
  "name": "",
  "item": "",
  "price": "",
  "date": ""
}
"""

response = llm.invoke(query)
print(response.content)

```json
{
  "name": "John",
  "item": "laptop",
  "price": "$1200",
  "date": "5th May 2026"
}
```


In [12]:
# Lists
query = """
List 5 programming languages.

Return as a numbered list only.
"""

response = llm.invoke(query)
print(response.content)

1. Java
2. Python
3. C++
4. JavaScript
5. Ruby


In [13]:
# Structured text
query = """
Summarize the topic "Linked List" in 100 words in this format:

Title:
Definition:
Applications:
Advantages:
"""

response = llm.invoke(query)
print(response.content)

Title: Linked List
Definition: A dynamic data structure consisting of nodes with data and pointers to next nodes.
Applications: Database query results, dynamic memory allocation, and browser history management.
Advantages: Efficient insertion/deletion, flexible memory usage, and good cache performance, making it suitable for applications with frequent data modifications.


### Self-Checking prompts

The model generates an answer, then critiques and improves its own response.
- Introduces self-evaluation
- Often improves quality

In [14]:
import sys
sys.path.append('Track1/Functions')
from C1_2_Func import self_checking_prompt

task = "Compute square root of 16."
print(self_checking_prompt(task, llm))

Initial Answer:
The square root of 16 is 4.

Critique:
The initial answer seems straightforward and correct, as 4 multiplied by 4 indeed equals 16. However, it's essential to consider the possibility of a negative square root, since (-4) * (-4) also equals 16. The answer should acknowledge both the positive and negative square roots.

Final Answer:
The square roots of 16 are 4 and -4, as both 4 * 4 and (-4) * (-4) equal 16.


### Iterative refinement

The model improves its output over multiple iterations.
- Multi-step improvement loop
- Simulates deeper thinking

In [15]:
from C1_2_Func import iterative_refinement

print(iterative_refinement("Explain dynamic programming in 100 words in simple terms.", llm))

Here's a revised answer in 100 words:

Dynamic programming is a problem-solving method that simplifies complex issues by breaking them down into smaller, solvable parts. It saves solutions to these smaller problems, avoiding repeated work and boosting efficiency. Think of it like a cookbook: once you've cooked a recipe, you can reuse it instead of starting over. Similarly, dynamic programming solves complex problems by storing and reusing solutions to smaller ones, saving time and effort. This approach makes complex problems more manageable, allowing for faster and more efficient solutions to be found. It's a smart way to tackle tough problems.


### Prompt Versioning

Prompts are treated like code and improved over time with version tracking.
- Maintain v1, v2, v3...
- Track performance changes with eval scores

In [16]:
prompt_versions = {
    "v1": """You are a customer support assistant for an e-commerce store.
Respond to the customer request.

Customer message: {input}

Return:
Response:
Next Step:""",
    "v2": """You are a customer support assistant for an e-commerce store.
Guidelines:
- Be empathetic and concise.
- If you need more information, ask one clarifying question.
- Provide a concrete next step.

Customer message: {input}

Return:
Response:
Next Step:""",
    "v3": """You are a customer support assistant for an e-commerce store.
Guidelines:
- Be empathetic and concise (max 3 sentences in Response).
- Acknowledge the issue or request.
- Provide one actionable next step.
- If critical info is missing, ask one clarifying question in Next Step.
- If order-related, request the order number.

Customer message: {input}

Return:
Response:
Next Step:""",
}

# Add v4+ as an exercise and compare scores after evaluation.

### Multi-Example Testing

Test prompts across diverse inputs to ensure robustness.
- Avoid overfitting to one example
- Identify edge cases early

In [17]:
test_inputs = [
    "My order #12345 was supposed to arrive yesterday. Can you check the status?",
    "I received a damaged mug. How do I get a replacement?",
    "I want to return a jacket bought last week. What's the return process?",
    "I was charged twice for my order. Please help.",
    "I need to change the shipping address for order #98765.",
    "The promo code SPRING20 isn't working at checkout.",
    "Can I cancel my order? It hasn't shipped yet.",
    "I forgot my password and can't log in.",
    "Do you ship internationally to Canada?",
    "I need an invoice for my last purchase.",
]

### Scoring prompts using LLM

In [18]:
from C1_2_Func import run_prompt_on_dataset

In [19]:
from C1_2_Func import llm_judge

### Testing Harness

Prompts are tested across diverse inputs to ensure robustness.
- Avoid overfitting to one example
- Identify edge cases

In [20]:
from C1_2_Func import test_harness

In [21]:
prompt_v1 = prompt_versions["v1"]

results = test_harness(prompt_v1, test_inputs, llm)

for r in results:
    print("\n---")
    print("Input:", r["input"])
    print("Score:", r["score"])


---
Input: My order #12345 was supposed to arrive yesterday. Can you check the status?
Score: 8.5

---
Input: I received a damaged mug. How do I get a replacement?
Score: 8.5

---
Input: I want to return a jacket bought last week. What's the return process?
Score: 8.8

---
Input: I was charged twice for my order. Please help.
Score: 8.5

---
Input: I need to change the shipping address for order #98765.
Score: 8.5

---
Input: The promo code SPRING20 isn't working at checkout.
Score: 8.5

---
Input: Can I cancel my order? It hasn't shipped yet.
Score: 8.5

---
Input: I forgot my password and can't log in.
Score: 9.0

---
Input: Do you ship internationally to Canada?
Score: 9.5

---
Input: I need an invoice for my last purchase.
Score: 8.5


### Failure testing

Identify and analyze cases where the model performs poorly.
- Focus on weaknesses, not just averages
- Drives prompt improvement

In [22]:
from C1_2_Func import get_failures

failures = get_failures(results)

for f in failures:
    print("\nFAILED CASE:")
    print("Input:", f["input"])
    print("Score:", f["score"])


FAILED CASE:
Input: My order #12345 was supposed to arrive yesterday. Can you check the status?
Score: 8.5

FAILED CASE:
Input: I received a damaged mug. How do I get a replacement?
Score: 8.5

FAILED CASE:
Input: I was charged twice for my order. Please help.
Score: 8.5

FAILED CASE:
Input: I need to change the shipping address for order #98765.
Score: 8.5

FAILED CASE:
Input: The promo code SPRING20 isn't working at checkout.
Score: 8.5

FAILED CASE:
Input: Can I cancel my order? It hasn't shipped yet.
Score: 8.5

FAILED CASE:
Input: I need an invoice for my last purchase.
Score: 8.5


### Evaluation and Comparison

Compare different prompt versions using scores of outputs.
- Evidence-based improvement
- Use metrics, not intuition

In [23]:
prompt_v2 = prompt_versions["v2"]

results_v2 = test_harness(prompt_v2, test_inputs, llm)

prompt_v3 = prompt_versions["v3"]

results_v3 = test_harness(prompt_v3, test_inputs, llm)

In [25]:
from C1_2_Func import average_score

V1 Avg: 8.68
V2 Avg: 8.5
V3 Avg: 8.5


In [26]:
version_log = [
    {
        "version": "v1",
        "prompt": prompt_versions["v1"],
        "avg_score": average_score(results),
        "failures": len(get_failures(results)),
        "notes": "Baseline"
    },
    {
        "version": "v2",
        "prompt": prompt_versions["v2"],
        "avg_score": average_score(results_v2),
        "failures": len(get_failures(results_v2)),
        "notes": "Clearer instructions"
    },
    {
        "version": "v3",
        "prompt": prompt_versions["v3"],
        "avg_score": average_score(results_v3),
        "failures": len(get_failures(results_v3)),
        "notes": "Structured constraints"
    },
]

for v in version_log:
    print(v)

{'version': 'v1', 'prompt': 'You are a customer support assistant for an e-commerce store.\nRespond to the customer request.\n\nCustomer message: {input}\n\nReturn:\nResponse:\nNext Step:', 'avg_score': 8.68, 'failures': 7, 'notes': 'Baseline'}
{'version': 'v2', 'prompt': 'You are a customer support assistant for an e-commerce store.\nGuidelines:\n- Be empathetic and concise.\n- If you need more information, ask one clarifying question.\n- Provide a concrete next step.\n\nCustomer message: {input}\n\nReturn:\nResponse:\nNext Step:', 'avg_score': 8.5, 'failures': 10, 'notes': 'Clearer instructions'}
{'version': 'v3', 'prompt': 'You are a customer support assistant for an e-commerce store.\nGuidelines:\n- Be empathetic and concise (max 3 sentences in Response).\n- Acknowledge the issue or request.\n- Provide one actionable next step.\n- If critical info is missing, ask one clarifying question in Next Step.\n- If order-related, request the order number.\n\nCustomer message: {input}\n\nRet

### Score Summary (Prompt Versions)

| Version | Avg Score | Failures | Notes |
| --- | --- | --- | --- |
| v1 | 8.6 | 0 | Baseline |
| v2 | 8.5 | 0 | Clearer instructions |
| v3 | 8.5 | 0 | Structured constraints |